# 1교시. 한국 영수증으로 구분하는 OCR·VLM·Document AI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/01_document_ai_overview.ipynb)

**이번 교시 행동:** 실제 영수증과 최종 Excel 사이의 역할 네 가지를 직접 연결합니다.

**통과 증거:** `course_outputs/receipt_pipeline_trace.json`

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

화면의 실행 모드를 먼저 확인합니다.

- `LIVE`: 현재 파일에 실제 모델을 실행한 결과
- `PREPARED_FALLBACK`: 공개 샘플을 사람이 검수해 둔 복구 결과
- 3분 이상 멈추면 실행을 중지하고 복구 결과로 계속합니다.
- 각 교시 끝에서 `CHECKPOINT PASS`와 산출물 파일을 확인합니다.


In [ ]:
import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
VALIDATION_MODE = os.getenv("COURSE_VALIDATE_PREPARED") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or VALIDATION_MODE:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 준비 입력을 쓰려면 "
            "USE_PREPARED_INPUT=True로 바꾸세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if VALIDATION_MODE:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())


In [ ]:
import base64
import io
from PIL import Image

GOLDEN_IMAGE_BASE64 = '...'
receipt_image = Image.open(
    io.BytesIO(base64.b64decode(GOLDEN_IMAGE_BASE64))
).convert("RGB")
receipt_image.thumbnail((720, 900))
receipt_image


## 기억할 네 문장

- **OCR**은 문서 이미지에서 글자와 위치를 읽습니다.
- **VLM**은 이미지와 언어를 함께 보고 문서의 의미·구조 초안을 만듭니다.
- **Document AI**는 분류·읽기·구조화·검증을 문서 처리 능력으로 묶습니다.
- **IDP**는 사람 승인, 예외 처리, 업무 시스템 연결, 운영 개선까지 포함합니다.

이 네 용어는 경쟁 제품 이름이 아니라 **포함 범위가 넓어지는 관계**입니다.
`OCR → VLM`은 반드시 거치는 고정 순서가 아닙니다.


## 내가 직접 채우는 4줄

네 역할이 남기는 산출물을 연결합니다. 모르면 빈칸으로 실행한 뒤
바로 아래 힌트·전체 정답과 비교합니다.


In [ ]:
# TODO: None 네 곳을 채우세요.
my_role_map = {
    "OCR": None,
    "VLM": None,
    "Document AI": None,
    "IDP": None,
}
print("내 연결:", my_role_map)


<details>
<summary>힌트와 전체 정답 보기</summary>

읽은 결과, 구조 초안, 검증 결과, 업무 파일 순서로 연결합니다.
</details>


In [ ]:
ANSWER_ROLE_MAP = {
    "OCR": "ocr_result.json",
    "VLM": "receipt_draft.json",
    "Document AI": "validated_receipt.json",
    "IDP": "receipt_result.xlsx",
}
assert len(ANSWER_ROLE_MAP) == 4
print("전체 정답:", ANSWER_ROLE_MAP)


In [ ]:
pipeline_trace = {
    "source_document": "taebaek_restaurant_2025_redacted.png",
    "input_policy": "approved_redacted_public_sample",
    "roles": [
        {"role": "OCR", "action": "글자·위치 판독", "artifact": "ocr_result.json"},
        {"role": "VLM", "action": "문서 의미·구조 초안", "artifact": "receipt_draft.json"},
        {"role": "Document AI", "action": "스키마·근거·규칙 검증", "artifact": "validated_receipt.json"},
        {"role": "IDP", "action": "사람 승인 후 Excel 연결", "artifact": "receipt_result.xlsx"},
    ],
    "evidence_example": {
        "field": "total_amount",
        "value": 76000,
        "source_text": "합계 금액 76,000",
        "decision": "REVIEW_BEFORE_EXPORT",
    },
    "scope_note": "0~12 전체 지도는 참고용이며 오늘은 한 장 경로를 구현합니다.",
}
output_path = OUTPUT_DIR / "receipt_pipeline_trace.json"
output_path.write_text(
    json.dumps(pipeline_trace, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
assert len(pipeline_trace["roles"]) == 4
print("CHECKPOINT 1/1 PASS:", output_path)
